In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("Hotel_Booking_Master.csv")

In [3]:
print(df.info())
print(df.head())
print(df.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10020 entries, 0 to 10019
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Date        10020 non-null  object
 1   Year        10020 non-null  int64 
 2   Month       10020 non-null  object
 3   Day         10020 non-null  int64 
 4   Weekday     10020 non-null  object
 5   Room_No     10020 non-null  int64 
 6   Guest_Name  2619 non-null   object
 7   Status      10020 non-null  object
dtypes: int64(3), object(5)
memory usage: 626.4+ KB
None
         Date  Year Month  Day   Weekday  Room_No Guest_Name  Status
0  2021-07-08  2021  July    8  Thursday        1        NaN  Vacant
1  2021-07-08  2021  July    8  Thursday        2        NaN  Vacant
2  2021-07-08  2021  July    8  Thursday        3        NaN  Vacant
3  2021-07-08  2021  July    8  Thursday        4        NaN  Vacant
4  2021-07-08  2021  July    8  Thursday        5        NaN  Vacant
(10020, 8)


In [4]:
df = df.drop_duplicates()

In [5]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

print(df.columns)

Index(['date', 'year', 'month', 'day', 'weekday', 'room_no', 'guest_name',
       'status'],
      dtype='object')


In [6]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")

df = df[df["date"].notna()]

In [7]:
df["guest_name"] = (
    df["guest_name"]
      .astype(str)
      .str.strip()
      .str.title()
)

df["guest_name"] = df["guest_name"].replace(
    [
        "",
        "Nan",
        "None",
        "0",
        "0.0",
        "-",
        "Vacant"
    ],
    pd.NA
)

In [8]:
df["room_no"] = (
    df["room_no"]
      .astype(str)
      .str.extract(r"(\d+)")
)

df["room_no"] = pd.to_numeric(
    df["room_no"],
    errors="coerce"
)

df = df[df["room_no"].isin([1,2,3,4,5,6])]

In [9]:
df["status"] = df["guest_name"].apply(
    lambda x: "Occupied" if pd.notna(x) else "Vacant"
)

In [10]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month_name()
df["month_no"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["weekday"] = df["date"].dt.day_name()
df["quarter"] = df["date"].dt.quarter

df["is_weekend"] = df["weekday"].isin(
    ["Saturday","Sunday"]
)

In [11]:
df = df.sort_values(
    ["date","room_no"]
).reset_index(drop=True)

In [12]:
print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nRoom Numbers")
print(df["room_no"].unique())

print("\nStatus")
print(df["status"].value_counts())

print("\nDate Range")
print(df["date"].min())
print(df["date"].max())


Missing Values
date             0
year             0
month            0
day              0
weekday          0
room_no          0
guest_name    7220
status           0
month_no         0
quarter          0
is_weekend       0
dtype: int64

Duplicate Rows
0

Room Numbers
[1 2 3 4 5 6]

Status
status
Vacant      7220
Occupied    2619
Name: count, dtype: int64

Date Range
2021-07-08 00:00:00
2026-06-30 00:00:00


In [13]:
df["guest_name"] = (
    df["guest_name"]
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)

In [14]:
df.head()

,date,year,month,day,weekday,room_no,guest_name,status,month_no,quarter,is_weekend
0,2021-07-08,2021,July,8,Thursday,1,<NA>,Vacant,7,3,False
1,2021-07-08,2021,July,8,Thursday,2,<NA>,Vacant,7,3,False
2,2021-07-08,2021,July,8,Thursday,3,<NA>,Vacant,7,3,False
3,2021-07-08,2021,July,8,Thursday,4,<NA>,Vacant,7,3,False
4,2021-07-08,2021,July,8,Thursday,5,<NA>,Vacant,7,3,False


In [16]:
df = df[
    ~df["guest_name"]
      .fillna("")
      .str.strip()
      .str.lower()
      .eq("out of service")
]

In [18]:
print(df["guest_name"].str.contains("Out of Service", case=False, na=False).sum())

0


In [19]:
df.to_csv(
    "Hotel_Booking_Clean_1.csv",
    index=False
)

print("Cleaning Completed Successfully!")

Cleaning Completed Successfully!


In [20]:
from sqlalchemy import create_engine

username = "root"
password = "########"
host = "localhost"
port = "3306"
database = "hotelbookings"

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")

# Write DataFrame to MySQL
table_name = "booking"   # choose any table name
df.to_sql(table_name, engine, if_exists="replace", index=False)

# Read back sample
pd.read_sql("SELECT * FROM booking LIMIT 5;", engine)

,date,year,month,day,weekday,room_no,guest_name,status,month_no,quarter,is_weekend
0,2021-07-08,2021,July,8,Thursday,1,None,Vacant,7,3,0
1,2021-07-08,2021,July,8,Thursday,2,None,Vacant,7,3,0
2,2021-07-08,2021,July,8,Thursday,3,None,Vacant,7,3,0
3,2021-07-08,2021,July,8,Thursday,4,None,Vacant,7,3,0
4,2021-07-08,2021,July,8,Thursday,5,None,Vacant,7,3,0
